# 04 - Gold Layer
## 01 - Gold Outputs

### Goal
- derive business indicators from silver integrated table
- silver_integrated is at grid event granularity - aggregate to region/day in gold
- compute real price volatility from raw silver market prices
- build gold tables and views using SQL
- log all outputs

In [0]:
%run ../01_setup/00_config

## Register source tables as temp views

In [0]:
from pyspark.sql import functions as F

integrated_df     = spark.table(silver_integrated_table)
market_prices_df  = spark.table(silver_market_prices_table)

integrated_df.createOrReplaceTempView("silver_integrated_view")
market_prices_df.createOrReplaceTempView("silver_market_prices_view")

print(f"silver_integrated rows:    {integrated_df.count()} (grid event granularity)")
print(f"silver_market_prices rows: {market_prices_df.count()} (one row per event_date/region/market_type)")

## Gold table - market volatility summary

Computed from raw silver market prices - one row per `event_date`/`region`.
Volatility is the standard deviation across `DAY_AHEAD`, `INTRADAY`, and `BALANCING` prices within each region/day.

In [0]:
spark.sql(f"""
CREATE OR REPLACE TABLE {gold_market_volatility_table} AS
WITH price_stats AS (
    SELECT
        event_date,
        region,
        ROUND(AVG(price_eur_mwh), 2)    AS avg_price_eur_mwh,
        ROUND(MIN(price_eur_mwh), 2)    AS min_price_eur_mwh,
        ROUND(MAX(price_eur_mwh), 2)    AS max_price_eur_mwh,
        ROUND(STDDEV(price_eur_mwh), 4) AS price_volatility,
        ROUND(SUM(volume_mwh), 2)       AS total_volume_mwh
    FROM silver_market_prices_view
    GROUP BY event_date, region
)
SELECT
    *,
    ROUND(max_price_eur_mwh - min_price_eur_mwh, 2) AS price_range,
    CASE
        WHEN price_volatility > 10 THEN 'HIGH'
        WHEN price_volatility > 5  THEN 'MEDIUM'
        ELSE 'LOW'
    END AS volatility_band
FROM price_stats
ORDER BY event_date, region
""")

print(f"Gold table saved: {gold_market_volatility_table}")
display(spark.table(gold_market_volatility_table).limit(10))

## Gold table - regional operations summary

Aggregates grid events to one row per `event_date`/`region`.
Joins market volatility to bring in real price metrics.

In [0]:
spark.sql(f"""
CREATE OR REPLACE TABLE {gold_regional_operations_table} AS
SELECT
    i.event_date,
    i.region,
    FIRST(i.country_code)                     AS country_code,
    FIRST(i.operating_zone)                   AS operating_zone,

    -- market indicators from volatility table
    v.avg_price_eur_mwh,
    v.total_volume_mwh,
    v.price_volatility,
    v.price_range,
    v.volatility_band,
    FIRST(i.has_high_price)                   AS has_high_price,

    -- weather indicators - already daily aggregates
    FIRST(i.avg_temperature_c)                AS avg_temperature_c,
    FIRST(i.avg_wind_speed_kmh)               AS avg_wind_speed_kmh,
    FIRST(i.total_precipitation_mm)           AS total_precipitation_mm,
    FIRST(i.has_severe_weather)               AS has_severe_weather,
    FIRST(i.wind_class)                       AS wind_class,

    -- event severity
    SUM(i.is_critical_event)                  AS critical_event_count,
    COUNT(i.event_id)                         AS total_events,
    ROUND(AVG(i.duration_minutes), 2)         AS avg_event_duration_minutes,

    -- derived scores
    CASE
        WHEN FIRST(i.has_severe_weather) = 1 THEN 2
        WHEN FIRST(i.avg_wind_speed_kmh) >= 60 THEN 1
        ELSE 0
    END AS weather_impact_score,

    CASE
        WHEN SUM(i.is_critical_event) > 0 AND FIRST(i.has_severe_weather) = 1 THEN 'HIGH'
        WHEN SUM(i.is_critical_event) > 0 OR FIRST(i.has_severe_weather) = 1  THEN 'MEDIUM'
        ELSE 'LOW'
    END AS operational_risk

FROM silver_integrated_view i
LEFT JOIN {gold_market_volatility_table} v
    ON i.event_date = v.event_date AND i.region = v.region
GROUP BY
    i.event_date, i.region,
    v.avg_price_eur_mwh, v.total_volume_mwh,
    v.price_volatility, v.price_range, v.volatility_band
ORDER BY i.event_date, i.region
""")

print(f"Gold table saved: {gold_regional_operations_table}")
display(spark.table(gold_regional_operations_table).limit(10))
dbutils.jobs.taskValues.set("gold_row_count", str(spark.table(gold_regional_operations_table).count()))

## Gold view - dashboard ready summary

In [0]:
spark.sql(f"""
CREATE OR REPLACE VIEW {gold_dashboard_view} AS
SELECT
    event_date,
    region,
    country_code,
    operating_zone,
    avg_price_eur_mwh,
    price_volatility,
    price_range,
    volatility_band,
    total_volume_mwh,
    avg_temperature_c,
    avg_wind_speed_kmh,
    has_severe_weather,
    weather_impact_score,
    critical_event_count,
    total_events,
    operational_risk
FROM {gold_regional_operations_table}
""")

print(f"Gold view created: {gold_dashboard_view}")
display(spark.table(gold_dashboard_view).limit(10))

Databricks visualization. Run in Databricks to view.

## Logging summary

In [0]:
ops_count = spark.table(gold_regional_operations_table).count()
vol_count = spark.table(gold_market_volatility_table).count()

print("=" * 60)
print("GOLD LAYER - RUN SUMMARY")
print("=" * 60)
print(f"Source integrated:    {silver_integrated_table} - {integrated_df.count()} rows (grid event level)")
print(f"Source market prices: {silver_market_prices_table} - {market_prices_df.count()} rows")
print(f"Gold operations:      {gold_regional_operations_table} - {ops_count} rows")
print(f"Gold market:          {gold_market_volatility_table} - {vol_count} rows")
print(f"Gold view:            {gold_dashboard_view}")
print(f"Run date:             {spark.sql('SELECT current_date()').collect()[0][0]}")
print("=" * 60)